# 03 — Model & Training

Defines SyncAV-Net and trains it on the cached features.

The model is deliberately conventional (EfficientNet-B4 → Bi-GRU → cross-modal
attention → MLP). **The contribution is the objective, not the architecture**, so most
of this notebook is unremarkable and that is by design. The cell that matters is §3.4,
the sync-contrastive loss.

## 3.1 Config

In [ ]:
import json, math, time, random
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

FEAT_ROOT = Path("../data/feats/lavdf")   # <-- primary training corpus
RUNS_ROOT = Path("../data/runs"); RUNS_ROOT.mkdir(parents=True, exist_ok=True)

# quick sanity check before anything else runs on a wrong/empty FEAT_ROOT
n_npz = len(list(FEAT_ROOT.glob("*.npz"))) if FEAT_ROOT.exists() else 0
if n_npz == 0:
    raise FileNotFoundError(
        f"No .npz feature files found at {FEAT_ROOT.resolve()}. "
        "Did Notebook 02 finish extracting features to THIS exact path? "
        'Run `find . -name "*.npz" | head` in a terminal to locate them if unsure.')
print(f"Found {n_npz} cached feature files at {FEAT_ROOT.resolve()}")

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"


@dataclass
class Cfg:
    visual_dim: int = 1792
    audio_dim: int = 768
    d_model: int = 512
    gru_hidden: int = 256      # bidirectional -> 512
    gru_layers: int = 2
    n_heads: int = 8
    dropout: float = 0.4
    seq_len: int = 30
    proj_dim: int = 128
    label_smoothing: float = 0.1
    # --- the knobs the ablation turns ---
    lambda_sync: float = 0.5   # THE CONTRIBUTION. 0.0 disables it.
    lambda_adv: float = 0.1
    adv_sigma: float = 0.05
    sync_temperature: float = 0.07
    use_crossmodal: bool = True
    use_temporal: bool = True
    # --- optimisation ---
    epochs: int = 40
    batch_size: int = 64
    lr: float = 1e-4
    weight_decay: float = 1e-4
    max_shift: int = 8         # audio roll offset for the hard negative

cfg = Cfg()
print(cfg)

## 3.2 Splits — with leak assertions

This is the second place the project can silently go wrong. We split so that:

* **test methods are unseen in training** (method-disjoint), and
* **no subject appears on both sides** (subject-disjoint).

Then we *assert* both properties actually hold. A split that merely intends to be
disjoint is worth nothing; the assertions are what make it true.

In [ ]:
# LAV-DF has only ONE generation pipeline -- there is nothing to hold out as an
# unseen method. Leave this empty; the split below switches to a pure
# subject(-surrogate)-disjoint design in that case. Generalisation to unseen
# generators is tested via the cross-dataset transfer to FF++ in Notebook 04,
# not a method holdout here.
HELD_OUT_METHODS = set()
VAL_FRAC = 0.10

files = sorted(FEAT_ROOT.glob("*.npz"))
meta = []
for f in files:
    d = np.load(f, allow_pickle=True)
    meta.append({"file": f,
                 "label": int(d["label"]),
                 "subject": str(d["subject"]),
                 "method": str(d["method"])})
print(f"{len(meta)} cached clips")

rng = random.Random(SEED)

if HELD_OUT_METHODS:
    # --- design A: hold out specific generation methods entirely (FakeAVCeleb/FF++) ---
    test = [m for m in meta if m["label"] == 1 and m["method"] in HELD_OUT_METHODS]
    remaining = [m for m in meta if not (m["label"] == 1 and m["method"] in HELD_OUT_METHODS)]
    subjects = sorted({m["subject"] for m in remaining})
    rng.shuffle(subjects)

    n_test_subj = max(1, int(0.15 * len(subjects)))
    n_val_subj  = max(1, int(VAL_FRAC * len(subjects)))
    test_subjects = set(subjects[:n_test_subj])
    val_subjects  = set(subjects[n_test_subj:n_test_subj + n_val_subj])

    # real clips from test subjects go to test (so test has both classes)
    test += [m for m in remaining if m["subject"] in test_subjects and m["label"] == 0]
    val   = [m for m in remaining if m["subject"] in val_subjects]
    train = [m for m in remaining
             if m["subject"] not in val_subjects and m["subject"] not in test_subjects]
else:
    # --- design B: no methods to hold out (LAV-DF) -- pure subject(-surrogate)-disjoint
    # split. Both classes come from the SAME held-out subjects, since a LAV-DF
    # "subject" groups a real source video together with every fake derived
    # from it (via the metadata's `original` field) -- so a held-out subject
    # naturally carries both real and fake clips into test/val together.
    subjects = sorted({m["subject"] for m in meta})
    rng.shuffle(subjects)

    n_test_subj = max(1, int(0.15 * len(subjects)))
    n_val_subj  = max(1, int(VAL_FRAC * len(subjects)))
    test_subjects = set(subjects[:n_test_subj])
    val_subjects  = set(subjects[n_test_subj:n_test_subj + n_val_subj])

    test  = [m for m in meta if m["subject"] in test_subjects]
    val   = [m for m in meta if m["subject"] in val_subjects]
    train = [m for m in meta
             if m["subject"] not in val_subjects and m["subject"] not in test_subjects]

def summarise(name, split):
    labs = [m["label"] for m in split]
    print(f"{name:6s} n={len(split):6d}  real={labs.count(0):5d} fake={labs.count(1):5d}  "
          f"subjects={len({m['subject'] for m in split})}")

summarise("train", train); summarise("val", val); summarise("test", test)


In [ ]:
# ---------------- LEAK ASSERTIONS ----------------
tr_subj = {m["subject"] for m in train}
va_subj = {m["subject"] for m in val}
te_subj = {m["subject"] for m in test}

leak_tv = tr_subj & va_subj
leak_tt = tr_subj & te_subj
assert not leak_tv, f"SUBJECT LEAK train/val: {list(leak_tv)[:5]}"
assert not leak_tt, f"SUBJECT LEAK train/test: {list(leak_tt)[:5]}"

tr_methods = {m["method"] for m in train if m["label"] == 1}
te_methods = {m["method"] for m in test  if m["label"] == 1}
method_leak = tr_methods & te_methods
if HELD_OUT_METHODS:
    assert not method_leak, (
        f"METHOD LEAK: {method_leak} appears in BOTH train and test. "
        "The model can memorise the generator it is evaluated on -- results would be "
        "inflated and the method-disjoint claim false.")
else:
    print("  No methods held out -- single-generation-pipeline dataset (e.g. LAV-DF). "
          "Method-disjoint generalisation is NOT being tested by this split; rely on "
          "the FF++ cross-dataset transfer in Notebook 04 for that claim instead.")

assert len(test) > 50, "Test split too small to be meaningful."
assert {m["label"] for m in test} == {0, 1}, "Test split is single-class -- AUC undefined."

print("SPLIT INTEGRITY OK")
print(f"  train methods: {sorted(tr_methods)}")
print(f"  test  methods: {sorted(te_methods)}  <- unseen in training (only meaningful if HELD_OUT_METHODS is non-empty)")
print(f"  subject overlap: none")


In [ ]:
# Write the exact split (clip ids per partition) so it can be published and re-used.
import json
SPLIT_DIR = Path("../splits"); SPLIT_DIR.mkdir(exist_ok=True)
splits = {name: sorted(m["file"].stem for m in part)
          for name, part in (("train", train), ("val", val), ("test", test))}
(SPLIT_DIR / "lavdf_splits.json").write_text(json.dumps(splits))
print({k: len(v) for k, v in splits.items()}, "-> ../splits/lavdf_splits.json")

## 3.3 Dataset

For each clip we also produce a **time-shifted audio** copy. For *real* clips this
becomes the hard negative of the sync-contrastive loss (§3.4).

In [ ]:
class AVDataset(Dataset):
    def __init__(self, meta, cfg, train=True):
        self.meta, self.cfg, self.train = meta, cfg, train

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, i):
        m = self.meta[i]
        d = np.load(m["file"], allow_pickle=True)
        visual = torch.from_numpy(d["visual"]).float()
        audio  = torch.from_numpy(d["audio"]).float()

        shift = (random.randint(2, self.cfg.max_shift) if self.train
                 else self.cfg.max_shift)
        audio_shift = torch.roll(audio, shifts=shift, dims=0)

        label = int(d["label"])
        return {"visual": visual, "audio": audio,
                "audio_shift": audio_shift,
                "label": torch.tensor(label),
                "real_mask": torch.tensor(label == 0)}


def _worker_init_fn(worker_id):
    # DataLoader workers are forked and, by default, only get a reseeded torch
    # RNG (base_seed + worker_id) -- Python's `random` and numpy are NOT
    # reseeded per worker unless we do it ourselves. AVDataset.__getitem__
    # uses `random.randint` for the audio-shift augmentation, so without this,
    # worker processes silently inherit whatever `random` state existed at
    # fork time instead of anything tied to the run's actual seed.
    worker_seed = (torch.initial_seed() + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(meta, cfg, train, seed=None):
    """Build a DataLoader with an explicitly seeded shuffle generator.

    Previously this had no seed control of its own: shuffle order rode on
    whatever the global torch RNG state happened to be when the loader was
    iterated, which is NOT reproducibly tied to any particular "seed" once
    the loader is reused across multiple training calls (as the ablation /
    seed-variance sweeps in Notebook 04 do). Passing `seed` here, and
    rebuilding the loader per seed in that sweep, closes that gap.
    """
    g = None
    if train and seed is not None:
        g = torch.Generator()
        g.manual_seed(seed)
    return DataLoader(AVDataset(meta, cfg, train), batch_size=cfg.batch_size,
                      shuffle=train, drop_last=train, num_workers=4, pin_memory=True,
                      generator=g, worker_init_fn=_worker_init_fn)

train_loader = make_loader(train, cfg, True, seed=SEED)
val_loader   = make_loader(val,   cfg, False)
test_loader  = make_loader(test,  cfg, False)
print(f"batches -- train {len(train_loader)}  val {len(val_loader)}  test {len(test_loader)}")


## 3.4 The model, and the objective that is the point of the paper

Read the `sync_contrastive_loss` docstring. Everything else here is standard.

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, in_dim, cfg):
        super().__init__()
        self.cfg = cfg
        self.inp = nn.Linear(in_dim, cfg.d_model)
        self.gru = nn.GRU(cfg.d_model, cfg.gru_hidden, cfg.gru_layers,
                          batch_first=True, bidirectional=True,
                          dropout=cfg.dropout if cfg.gru_layers > 1 else 0.0)
        self.norm = nn.LayerNorm(2 * cfg.gru_hidden)
        self.out = nn.Linear(2 * cfg.gru_hidden, cfg.d_model)

    def forward(self, x):
        x = self.inp(x)
        if not self.cfg.use_temporal:          # ablation: skip the GRU entirely
            return x
        h, _ = self.gru(x)
        return self.out(self.norm(h))


class CrossModalBridge(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.v2a = nn.MultiheadAttention(cfg.d_model, cfg.n_heads,
                                         dropout=cfg.dropout, batch_first=True)
        self.a2v = nn.MultiheadAttention(cfg.d_model, cfg.n_heads,
                                         dropout=cfg.dropout, batch_first=True)
        self.proj = nn.Sequential(
            nn.Linear(2 * cfg.d_model, cfg.d_model), nn.LayerNorm(cfg.d_model),
            nn.GELU(), nn.Dropout(cfg.dropout))

    def forward(self, v, a):
        if not self.cfg.use_crossmodal:        # ablation: plain concat (late fusion)
            return self.proj(torch.cat([v, a], dim=-1))
        v_ctx, _ = self.v2a(v, a, a)           # visual queries audio
        a_ctx, _ = self.a2v(a, v, v)           # audio queries visual
        return self.proj(torch.cat([v_ctx, a_ctx], dim=-1))


class SyncAVNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.venc = TemporalEncoder(cfg.visual_dim, cfg)
        self.aenc = TemporalEncoder(cfg.audio_dim, cfg)
        self.bridge = CrossModalBridge(cfg)
        self.det = nn.Sequential(
            nn.Linear(cfg.d_model, 256), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(256, 2))
        self.gv = nn.Sequential(nn.Linear(cfg.d_model, cfg.proj_dim), nn.GELU(),
                                nn.Linear(cfg.proj_dim, cfg.proj_dim))
        self.ga = nn.Sequential(nn.Linear(cfg.d_model, cfg.proj_dim), nn.GELU(),
                                nn.Linear(cfg.proj_dim, cfg.proj_dim))

    def forward(self, visual, audio):
        v, a = self.venc(visual), self.aenc(audio)
        z = self.bridge(v, a)
        return {"logits": self.det(z.mean(1)), "v": v, "a": a}

    def project(self, v, a):
        return (F.normalize(self.gv(v.mean(1)), dim=-1),
                F.normalize(self.ga(a.mean(1)), dim=-1))


model = SyncAVNet(cfg).to(device)
n_par = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"trainable parameters: {n_par:.2f} M   (backbones are frozen & cached)")

In [ ]:
def sync_contrastive_loss(zv, za, zv_sh, za_sh, tau):
    """Symmetric InfoNCE over REAL clips only.

    THE KEY IDEA. For clip i the positive is (zv_i, za_i) -- its correctly aligned
    audio. The hard negative is (zv_i, za_sh_i) -- a TIME-SHIFTED copy of THE SAME
    CLIP's audio.

    Positive and hard negative therefore share speaker, microphone, room, lighting,
    codec, semantic content -- and every generation artefact. They differ in exactly
    one respect: temporal alignment. Consequently no artefact feature can reduce this
    loss (it takes the same value on both and cancels), and the only way to reduce it
    is to represent audio-visual SYNCHRONY.

    Synchrony is a property of authentic recording, not of any particular generator.
    That is why the representation this loss shapes should transfer across datasets.
    """
    N = zv.size(0)
    keys_a = torch.cat([za, za_sh], 0)                  # (2N, d): aligned + shifted
    logits_v = zv @ keys_a.t() / tau
    tgt = torch.arange(N, device=zv.device)
    loss_v = F.cross_entropy(logits_v, tgt)

    keys_v = torch.cat([zv, zv_sh], 0)
    logits_a = za @ keys_v.t() / tau
    loss_a = F.cross_entropy(logits_a, tgt)
    return 0.5 * (loss_v + loss_a)


def compute_losses(model, batch, cfg):
    out = model(batch["visual"], batch["audio"])
    l_det = F.cross_entropy(out["logits"], batch["label"],
                            label_smoothing=cfg.label_smoothing)
    parts = {"det": l_det}
    total = l_det

    rm = batch["real_mask"]
    if cfg.lambda_sync > 0 and rm.sum() >= 2:      # need >=2 reals for contrast
        v_r = out["v"][rm]
        a_r = out["a"][rm]
        a_sh = model.aenc(batch["audio_shift"][rm])
        zv, za = model.project(v_r, a_r)
        zv_sh, za_sh = model.project(v_r, a_sh)   # visual fixed; audio desynced
        l_sync = sync_contrastive_loss(zv, za, zv_sh, za_sh, cfg.sync_temperature)
        parts["sync"] = l_sync
        total = total + cfg.lambda_sync * l_sync

    if cfg.lambda_adv > 0:
        with torch.no_grad():
            base = out["logits"].softmax(-1)
        if random.random() < 0.5:
            pert = model(batch["visual"] + cfg.adv_sigma * torch.randn_like(batch["visual"]),
                         batch["audio"])["logits"].softmax(-1)
        else:
            pert = model(batch["visual"],
                         batch["audio"] + cfg.adv_sigma * torch.randn_like(batch["audio"])
                         )["logits"].softmax(-1)
        l_adv = F.mse_loss(pert, base)
        parts["adv"] = l_adv
        total = total + cfg.lambda_adv * l_adv

    parts["total"] = total
    return total, parts

## 3.5 Metrics

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    S, Y = [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        p = model(b["visual"], b["audio"])["logits"].softmax(-1)[:, 1]
        S.append(p.cpu()); Y.append(b["label"].cpu())
    if not S:
        return {}
    s = torch.cat(S).numpy(); y = torch.cat(Y).numpy()

    pred = (s >= 0.5).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    prec = tp / max(1, tp + fp); rec = tp / max(1, tp + fn)
    f1 = 2 * prec * rec / max(1e-9, prec + rec)
    acc = (tp + tn) / max(1, len(y))

    auc, eer = 0.0, 1.0
    if len(set(y.tolist())) > 1:
        auc = float(roc_auc_score(y, s))
        fpr, tpr, _ = roc_curve(y, s); fnr = 1 - tpr
        eer = float(fpr[np.nanargmin(np.abs(fnr - fpr))])

    return {"auc": round(auc, 4), "f1": round(f1, 4), "recall": round(rec, 4),
            "precision": round(prec, 4), "accuracy": round(acc, 4), "eer": round(eer, 4)}

## 3.6 Train

Model selection is on **validation AUC**. Roughly 30–60 min for 40 epochs on cached
features on an A5000.

In [ ]:
def train_model(cfg, train_loader, val_loader, test_loader, tag="full", seed=None, verbose=True):
    # FIX: previously this always did torch.manual_seed(SEED) -- the fixed global
    # constant, ignoring whatever seed the caller actually wanted. That meant model
    # weight initialisation was IDENTICAL on every call, regardless of which "seed"
    # a sweep (e.g. Notebook 04's seed-variance check) thought it was using. Any
    # apparent run-to-run variation was coming from elsewhere (see Notebook 04's
    # comments), not from genuinely different initialisation. `seed` now defaults
    # to the original global SEED so a plain `train_model(cfg, ...)` call behaves
    # exactly as before; pass `seed=s` explicitly to control it per call.
    use_seed = seed if seed is not None else SEED
    torch.manual_seed(use_seed)
    random.seed(use_seed)
    np.random.seed(use_seed)

    model = SyncAVNet(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg.lr, total_steps=cfg.epochs * max(1, len(train_loader)),
        pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

    out_dir = RUNS_ROOT / tag; out_dir.mkdir(parents=True, exist_ok=True)
    history, best_auc = [], -1.0

    for ep in range(cfg.epochs):
        model.train(); t0 = time.time()
        agg = {}
        for b in train_loader:
            b = {k: v.to(device, non_blocking=True) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                loss, parts = compute_losses(model, b, cfg)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            for k, v in parts.items():
                agg[k] = agg.get(k, 0.0) + float(v.detach())

        nb = max(1, len(train_loader))
        vm = evaluate(model, val_loader)
        rec = {"epoch": ep, "train_loss": {k: v / nb for k, v in agg.items()},
               "val": vm, "time_s": round(time.time() - t0, 1)}
        history.append(rec)
        if verbose:
            print(f"[{tag}][{ep:02d}] loss={agg.get('total',0)/nb:.4f} "
                  f"val_auc={vm['auc']:.4f} val_f1={vm['f1']:.4f} ({rec['time_s']}s)")

        if vm["auc"] > best_auc:
            best_auc = vm["auc"]
            torch.save({"model": model.state_dict(), "cfg": asdict(cfg)},
                       out_dir / "best.pt")

    ck = torch.load(out_dir / "best.pt", map_location=device)
    model.load_state_dict(ck["model"])
    test_m = evaluate(model, test_loader)
    print(f"[{tag}] TEST (method+subject disjoint): {test_m}")

    (out_dir / "history.json").write_text(json.dumps(
        {"history": history, "test": test_m, "config": asdict(cfg), "seed": use_seed}, indent=1))
    return model, test_m


model, test_metrics = train_model(cfg, train_loader, val_loader, test_loader, tag="full")


### Read the test numbers with suspicion

If in-dataset AUC is very high (>0.99), be suspicious rather than pleased. On a
genuinely method-disjoint split that would be surprising, and the likeliest explanation
is a split defect — go back and check the assertions in §3.2 actually ran.

The in-dataset number is **not** the paper's claim anyway. The claim lives in Notebook
04: how well this transfers to a corpus it has never seen.

In [ ]:
print("Saved:", RUNS_ROOT / "full" / "best.pt")
print("\nNext: 04_cross_dataset_and_ablations.ipynb  <- the experiment that matters")